# ProboSed — 04: Porewater Geochemistry Profiles

**IODP Expedition 405 — Site C0019, Japan Trench**

Produces three geochemistry figures connecting porewater anomalies to
MTD intervals identified from VCD observations:

1. **C0019J** — five-panel depth profile, 93-825 mbsf (deep hole to fault)
2. **C0019M** — five-panel depth profile, 0-107 mbsf (shallow SMTZ)
3. **Combined overview** — side-by-side CH4, SO4, Ca comparison

**Key observations:**
- SO4 and Ca anomaly at ~820 mbsf (fault zone fluid advection)
- SMTZ at ~40 mbsf in M hole establishes diagenetic baseline
- Ca contrast between M and J holes confirms deep signal is not diagenetic

**MTD integration:**
When an MTD catalog from `01_labeling.ipynb` is provided, MTD intervals
are shaded on every panel — directly linking geochemical anomalies to
sediment fabric observations.

In [ ]:
# Cell 1: Install dependencies
!pip install pandas openpyxl matplotlib scipy --quiet

In [ ]:
# Cell 2: Mount Drive and set up ProboSed
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
repo_url  = 'https://github.com/yourusername/ProboSed.git'  # update this
repo_path = '/content/ProboSed'
result = subprocess.run(['git', 'clone', repo_url, repo_path],
                        capture_output=True, text=True)
if result.returncode != 0:
    subprocess.run(['git', '-C', repo_path, 'pull'], capture_output=True)
sys.path.insert(0, repo_path)
print('ProboSed modules available')

In [ ]:
# Cell 3: Configure paths
# Update DATA_DIR to match the Google Drive folder containing the data files.
import os

DATA_DIR   = '/content/drive/MyDrive/iodp/X405/Data & Data Tracking'
OUTPUT_DIR = '/content/drive/MyDrive/iodp/X405/ProboSed_Output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Interstitial water summary sheets (.xlsx)
IW_J = os.path.join(DATA_DIR, 'SummarySheet-IW-Hole_Exp405_C0019J_250312.xlsx')
IW_M = os.path.join(DATA_DIR, 'SummarySheet-IW-Hole_Exp405_C0019M_250312.xlsx')

# Headspace gas summary sheets (.xlsm)
GC_J = os.path.join(DATA_DIR, 'SummarySheet-GC_200_160206_Exp405_C0019J.xlsm')
GC_M = os.path.join(DATA_DIR, 'SummarySheet-GC_200_160206_Exp405_C0019M.xlsm')

# MTD catalog from notebook 01 (optional — enables MTD shading on figures)
MTD_CSV = os.path.join(OUTPUT_DIR, 'C0019J_MTD_catalog.csv')

print(f'Data directory:   {DATA_DIR}')
print(f'Output directory: {OUTPUT_DIR}')

In [ ]:
# Cell 4: Load geochemistry data
from geochem.geochem_figures import load_iw, load_gc

print('Loading interstitial water data...')
iw_j = load_iw(IW_J)
iw_m = load_iw(IW_M)

print('Loading headspace gas data...')
gc_j = load_gc(GC_J, iw_j)
gc_m = load_gc(GC_M, iw_m)

print(f'C0019J: {len(iw_j)} IW samples, {len(gc_j)} GC samples')
print(f'C0019M: {len(iw_m)} IW samples, {len(gc_m)} GC samples')
iw_j.head()

In [ ]:
# Cell 5: Load MTD catalog (optional)
# If the catalog CSV from notebook 01 exists, load it to enable MTD shading.
# If it does not exist, figures are produced without MTD interval overlays.
import pandas as pd

if os.path.exists(MTD_CSV):
    mtd_catalog = pd.read_csv(MTD_CSV)
    print(f'MTD catalog loaded: {len(mtd_catalog)} intervals')
    print(mtd_catalog)
else:
    mtd_catalog = None
    print('MTD catalog not found — figures will be produced without MTD shading.')
    print(f'Run notebook 01 first and save output to: {MTD_CSV}')

## Quick data overview

Check the depth ranges and key values before plotting.

In [ ]:
# Cell 6: Data overview
import numpy as np

print('=== C0019J (deep hole) ===')
print(f'  Depth range:      {iw_j["depth_m"].min():.1f} - {iw_j["depth_m"].max():.1f} mbsf')
print(f'  SO4 range:        {iw_j["SO4"].min():.1f} - {iw_j["SO4"].max():.1f} mM')
print(f'  Ca range:         {iw_j["Ca"].min():.1f} - {iw_j["Ca"].max():.1f} mM')
print(f'  Chlorinity range: {iw_j["chlorinity"].min():.0f} - {iw_j["chlorinity"].max():.0f} mM')
print(f'  CH4 range:        {gc_j["Methane"].min():.0f} - {gc_j["Methane"].max():.0f} ppm')

print('\n=== C0019M (shallow hole) ===')
print(f'  Depth range:      {iw_m["depth_m"].min():.1f} - {iw_m["depth_m"].max():.1f} mbsf')
print(f'  SO4 range:        {iw_m["SO4"].min():.1f} - {iw_m["SO4"].max():.1f} mM')
smtz = iw_m[iw_m['SO4'] < 1]['depth_m'].min()
print(f'  SMTZ (SO4 < 1 mM): {smtz:.1f} mbsf')

# key fault zone result
fault_ca = iw_j[iw_j['depth_m'] > 820]['Ca'].values
bg_ca    = iw_j[(iw_j['depth_m'] > 600) & (iw_j['depth_m'] < 800)]['Ca'].mean()
if len(fault_ca):
    print(f'\n=== Fault zone Ca enrichment ===')
    print(f'  Ca at fault zone (~825 mbsf): {fault_ca[0]:.1f} mM')
    print(f'  Ca background (600-800 mbsf): {bg_ca:.1f} mM')
    print(f'  Enrichment factor:            {fault_ca[0]/bg_ca:.1f}x')

## Figures

All three figures are produced below.
If `mtd_catalog` was loaded in Cell 5, MTD intervals are shaded on every panel.

In [ ]:
# Cell 7: Figure 1 — C0019J deep hole profile
from geochem.geochem_figures import plot_C0019J

out_j = os.path.join(OUTPUT_DIR, 'C0019J_geochemistry_profiles.png')
plot_C0019J(iw_j, gc_j, out_j, mtd_catalog=mtd_catalog)

# display inline
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
img = mpimg.imread(out_j)
plt.figure(figsize=(16, 10))
plt.imshow(img)
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 8: Figure 2 — C0019M shallow hole profile
from geochem.geochem_figures import plot_C0019M

out_m = os.path.join(OUTPUT_DIR, 'C0019M_geochemistry_profiles.png')
plot_C0019M(iw_m, gc_m, out_m, mtd_catalog=mtd_catalog)

img = mpimg.imread(out_m)
plt.figure(figsize=(16, 8))
plt.imshow(img)
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 9: Figure 3 — combined overview
from geochem.geochem_figures import plot_combined

out_combined = os.path.join(OUTPUT_DIR, 'C0019_combined_overview.png')
plot_combined(iw_j, gc_j, iw_m, gc_m, out_combined, mtd_catalog=mtd_catalog)

img = mpimg.imread(out_combined)
plt.figure(figsize=(14, 12))
plt.imshow(img)
plt.axis('off')
plt.tight_layout()
plt.show()

## Key quantitative results

These values are primary observations — record them with the run date.

In [ ]:
# Cell 10: Print key quantitative results
print('Key quantitative results:')
print(f'  Ca at fault zone (~825 mbsf): {fault_ca[0]:.1f} mM')
print(f'  Ca background mean (600-800 mbsf): {bg_ca:.1f} mM')
print(f'  Ca enrichment factor at fault: {fault_ca[0]/bg_ca:.1f}x')
print(f'  SMTZ in C0019M (SO4 < 1 mM): {smtz:.1f} mbsf')
max_ch4_idx = gc_m['Methane'].idxmax()
print(f'  Peak CH4 in C0019M: {gc_m.loc[max_ch4_idx, "Methane"]:.0f} ppm '
      f'at {gc_m.loc[max_ch4_idx, "depth_m"]:.1f} mbsf')

if mtd_catalog is not None:
    print(f'  MTD intervals overlaid: {len(mtd_catalog)}')
    for _, row in mtd_catalog.iterrows():
        print(f'    {row["mtd_id"]}: {row["top_m"]:.1f} - {row["bottom_m"]:.1f} m '
              f'({row["thickness_m"]:.1f} m thick, mean stability = {row["mean_stability"]:.1f})')